# OW v3 additive_reference_ow_v3

Fresh ImageNet B7 experiment on Wild with completed OW00 v7 pseudo masks.
Architecture: `{"color_srm": false, "decoder": "dense", "frequency": "global", "fusion": "late_additive", "implementation_version": "ow_v3", "multi_stream": true, "pooling": "avg", "segmentation": true}`.

Attach `ahmedgamal/openforensics-wild-dataset` and the completed OW00 version7 output.
For continuation, attach only this experiment's latest full output. Smoke/full and
source/protocol mismatches are rejected. Do not load legacy teacher or v2 weights.
Default smoke trains one epoch and checks Validation only. Full runs train three
epochs, one per saved run, select by Validation accuracy, and evaluate Test once
training is complete. Binary masks are teacher targets, not independent GT.
See the GitHub notebooks-ow-v3 README for changes, ablations and limitations.


## 1. Environment


In [ ]:
import importlib
import subprocess
import sys

def _pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

missing = []
for mod, pip_name in {
    "cv2": "opencv-python-headless",
    "sklearn": "scikit-learn",
    "pandas": "pandas",
    "tqdm": "tqdm",
}.items():
    try:
        importlib.import_module(mod)
    except Exception:
        missing.append(pip_name)
if missing:
    _pip_install(missing)

import time
import torch
NOTEBOOK_START_TIME = time.time()
print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Imports


In [ ]:
import gc
import hashlib
import json
import math
import os
import random
import shutil
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from torch.utils.data import DataLoader, Dataset
from torchvision.models import EfficientNet_B7_Weights, efficientnet_b7
from tqdm.auto import tqdm


## 3. Wild images and the generated-mask folder


In [ ]:
def discover_wild_root() -> Path:
    roots = []
    if Path("/kaggle/input").exists():
        roots.append(Path("/kaggle/input"))
    roots.extend([Path.cwd() / "openforensics-wild-dataset", Path.cwd() / "Dataset", Path.cwd()])
    for root in roots:
        if not root.exists():
            continue
        for dirpath, dirnames, _ in os.walk(root):
            p = Path(dirpath)
            if "ow_masks" in p.parts:
                continue
            kids = {d.lower() for d in dirnames}
            if {"train", "validation", "test"} <= kids or {"train", "val", "test"} <= kids:
                return p
            if p.name.lower() == "dataset" and ("train" in kids or "test" in kids):
                return p
    raise FileNotFoundError("Attach ahmedgamal/openforensics-wild-dataset.")


def _split_dir(root: Path, name: str) -> Path:
    aliases = {
        "train": ["Train", "train"],
        "val": ["Validation", "Val", "validation", "val"],
        "test": ["Test", "test"],
    }
    for a in aliases[name]:
        p = root / a
        if p.is_dir():
            return p
    raise FileNotFoundError(f"No {name} split under {root}")


WILD_ROOT = discover_wild_root()
IN_KAGGLE = Path("/kaggle/working").exists()
WORKING = Path("/kaggle/working") if IN_KAGGLE else Path.cwd() / "kaggle_working"
WORKING.mkdir(parents=True, exist_ok=True)
for sub in ("checkpoints", "predictions", "logs"):
    (WORKING / sub).mkdir(parents=True, exist_ok=True)
print("wild root:", WILD_ROOT)
print("working:", WORKING)


## 4. Configuration


In [ ]:
CONFIG = {'run_mode': 'smoke', 'seed': 42, 'implementation_version': 'ow_v3', 'use_pretrained': True, 'image_size': 600, 'batch_size': 4, 'num_workers': 2, 'epochs': 3, 'learning_rate': 0.0001, 'weight_decay': 0.0001, 'classification_threshold': 0.5, 'mask_threshold': 0.5, 'lambda_cls': 1.0, 'lambda_seg': 0.25, 'dice_weight': 1.0, 'use_amp': True, 'gradient_accumulation': 1, 'skip_training': False, 'resume': True, 'resume_from_input': True, 'time_budget_hours': 11.0, 'epochs_per_run': 1, 'mid_epoch_checkpoint_minutes': 30, 'experiment': 'additive_reference_ow_v3', 'max_per_split': None, 'backbone_lr_multiplier': 0.1, 'warmup_fraction': 0.05, 'freeze_backbone_bn': True, 'seg_quality_csv': None, 'benchmark_inference': True, 'source_code_hash': '112357f498eda8ed0e637072fe86b9c4715eaa6b17a173b3a2a46016dc1dc6f2'}
EXP_CFG = {'multi_stream': True, 'segmentation': True, 'fusion': 'late_additive', 'color_srm': False, 'frequency': 'global', 'pooling': 'avg', 'decoder': 'dense', 'implementation_version': 'ow_v3'}
if os.name == "nt":
    CONFIG["num_workers"] = 0
if CONFIG["run_mode"] not in ("smoke", "full"):
    raise ValueError("run_mode must be smoke or full")
if CONFIG["run_mode"] == "smoke":
    CONFIG.update(max_per_split=64, epochs=1)
if CONFIG["run_mode"] == "full" and not torch.cuda.is_available():
    raise RuntimeError("Full training requires a Kaggle GPU")
if int(CONFIG["gradient_accumulation"]) < 1:
    raise ValueError("gradient_accumulation must be positive")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("EXPERIMENT", CONFIG["experiment"], EXP_CFG, CONFIG["run_mode"])


## 5. Join each Wild image to its generated mask


In [ ]:
EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
LABELS = {"real": 0, "fake": 1}


def rel_key(split: str, class_name: str, file_name: str) -> str:
    return f"{split}/{class_name}/{file_name}"


def _class_dir(split_dir: Path, name: str):
    for a in (name, name.capitalize(), name.upper(), name.lower()):
        p = split_dir / a
        if p.is_dir():
            return p
    return None


def list_images(split_name: str) -> pd.DataFrame:
    split_dir = _split_dir(WILD_ROOT, split_name)
    rows = []
    for cname, lab in LABELS.items():
        cdir = _class_dir(split_dir, cname)
        if cdir is None:
            print("WARNING: missing", split_name, cname)
            continue
        for p in sorted(cdir.iterdir()):
            if p.is_file() and p.suffix.lower() in EXTS:
                rows.append({
                    "path": str(p),
                    "split": split_name,
                    "label": lab,
                    "class_name": cname,
                    "file_name": p.name,
                    "rel_key": rel_key(split_name, cname, p.name),
                })
    return pd.DataFrame(rows)


def unpack_mask_outputs(input_root: Path) -> list:
    import tempfile
    import zipfile
    from pathlib import PurePosixPath

    roots = []
    for archive in input_root.rglob("_output_.zip"):
        with zipfile.ZipFile(archive) as z:
            members = [m for m in z.infolist() if not m.is_dir()
                       and "ow_masks" in PurePosixPath(m.filename.replace("\\", "/")).parts]
            if not members:
                continue
            target = Path(tempfile.mkdtemp(prefix="veritas_ow_masks_"))
            for member in members:
                rel = PurePosixPath(member.filename.replace("\\", "/"))
                if rel.is_absolute() or ".." in rel.parts:
                    raise ValueError(f"Unsafe mask archive path: {member.filename}")
                rel = Path(*rel.parts[rel.parts.index("ow_masks"):])
                dst = target / rel
                if not dst.resolve().is_relative_to(target.resolve()):
                    raise ValueError(f"Unsafe mask archive path: {member.filename}")
                dst.parent.mkdir(parents=True, exist_ok=True)
                with z.open(member) as src, dst.open("wb") as out:
                    shutil.copyfileobj(src, out)
            roots.append(target)
            print("unpacked mask output", archive, "files", len(members), "into", target)
    return roots


def load_mask_table() -> pd.DataFrame:
    frames = []
    roots = []
    for p in (Path("/kaggle/input"), Path("/kaggle/working"), WORKING, Path.cwd()):
        if p.exists():
            roots.append(p)
    roots.extend(unpack_mask_outputs(Path("/kaggle/input")))
    seen = set()
    for root in roots:
        for dirpath, _, filenames in os.walk(root):
            if "index.csv" not in filenames:
                continue
            path = Path(dirpath) / "index.csv"
            key = str(path)
            if key in seen:
                continue
            seen.add(key)
            try:
                df = pd.read_csv(path, dtype=str)
            except Exception as e:
                print("skip", path, e)
                continue
            if not {"image_id", "rel_key", "mask_file", "status", "crop_x0"} <= set(df.columns):
                continue
            df["_root"] = str(path.parent)
            frames.append(df)
            print("mask index", path, "rows", len(df))
    if not frames:
        raise FileNotFoundError(
            "No ow_masks/index.csv found. Run OW_00_generate_masks and attach that notebook output."
        )
    all_df = pd.concat(frames, ignore_index=True)
    kept = []
    used = set()
    for rec in all_df.to_dict("records"):
        iid = str(rec["image_id"])
        if iid in used or str(rec.get("status")) != "ok":
            continue
        png = Path(str(rec["_root"])) / str(rec["mask_file"])
        if not png.is_file():
            continue
        rec["mask_path"] = str(png)
        kept.append(rec)
        used.add(iid)
    if not kept:
        raise FileNotFoundError("index.csv rows were found, but none of the PNG files are present.")
    out = pd.DataFrame(kept).drop_duplicates("rel_key", keep="last")
    print("masks ready", len(out), "teachers", sorted(set(out.teacher.dropna().astype(str))) if "teacher" in out.columns else [])
    return out


MASKS = load_mask_table()
INDEX = {}
for split in ("train", "val", "test"):
    images = list_images(split)
    if images.empty:
        raise RuntimeError(f"{split} has no images")
    merged = images.merge(
        MASKS[["rel_key", "mask_path", "crop_x0", "crop_y0", "crop_x1", "crop_y1", "status", "positive_frac"]],
        on="rel_key",
        how="left",
    )
    have = merged["mask_path"].notna()
    print(f"{split}: images {len(merged)}  with mask {int(have.sum())}  missing {int((~have).sum())}")
    if CONFIG["run_mode"] == "full" and (~have).any():
        missing = merged.loc[~have, "rel_key"].head(8).tolist()
        raise RuntimeError(
            f"{split} is missing {int((~have).sum())} generated masks. Finish OW_00 before full training. Example: {missing}"
        )
    ready = merged.loc[have].copy()
    if CONFIG.get("max_per_split"):
        parts = []
        rng = np.random.RandomState(CONFIG["seed"] + {"train": 1, "val": 2, "test": 3}[split])
        for _, g in ready.groupby("label"):
            n = min(int(CONFIG["max_per_split"]), len(g))
            parts.append(g.sample(n=n, random_state=rng) if n < len(g) else g)
        ready = pd.concat(parts)
    if ready.empty:
        raise RuntimeError(f"{split} has no generated masks. Run OW_00 first.")
    INDEX[split] = ready.reset_index(drop=True)
    print(f"  using {len(INDEX[split])}  fake={int((INDEX[split].label == 1).sum())} real={int((INDEX[split].label == 0).sum())}")



# Hash source data/targets and common training settings; architecture is separate.
assert len(MASKS) == 190335, "Attach the complete OW00 v7 masks"
quality_path = CONFIG.get("seg_quality_csv")
quality = None
if quality_path:
    quality = pd.read_csv(quality_path)
    if not {"rel_key", "seg_valid"} <= set(quality) or quality.rel_key.duplicated().any():
        raise ValueError("Quality CSV requires unique rel_key and binary seg_valid")
    if not quality.seg_valid.isin([0, 1]).all():
        raise ValueError("seg_valid must be binary, based on an external quality assessment")
    if quality.rel_key.str.startswith("test/").any():
        raise ValueError("Quality decisions must use Train/Validation, not Test")
for split, frame in INDEX.items():
    frame["seg_valid"] = 1.0
    if quality is not None and split != "test":
        flags = quality.set_index("rel_key").seg_valid
        frame["seg_valid"] = frame.rel_key.map(flags).fillna(1).astype(float)
mask_cols = ["rel_key", "label", "mask_file", "crop_x0", "crop_y0", "crop_x1", "crop_y1", "positive_frac", "teacher"]
mask_hash = hashlib.sha256(MASKS[mask_cols].sort_values("rel_key").to_csv(index=False).encode()).hexdigest()
index_hash = hashlib.sha256(pd.concat([INDEX[s][["rel_key", "label", "seg_valid"]].sort_values("rel_key").assign(split=s)
    for s in ("train", "val", "test")]).to_csv(index=False).encode()).hexdigest()
keys = ["run_mode", "seed", "epochs", "batch_size", "image_size", "learning_rate", "weight_decay", "use_pretrained",
        "classification_threshold", "mask_threshold", "lambda_cls", "lambda_seg", "dice_weight", "use_amp",
        "gradient_accumulation", "backbone_lr_multiplier", "warmup_fraction", "freeze_backbone_bn", "source_code_hash"]
protocol = {k: CONFIG[k] for k in keys}
protocol.update(mask_source_hash=mask_hash, data_index_hash=index_hash, selection_metric="val_accuracy",
                supervision_policy="real_zero_fake_nonempty_v3", augmentation="aligned_epoch_flip_v3")
PROTOCOL_HASH = hashlib.sha256(json.dumps(protocol, sort_keys=True).encode()).hexdigest()
(WORKING / "logs/protocol.json").write_text(json.dumps(dict(hash=PROTOCOL_HASH, protocol=protocol,
    experiment=CONFIG["experiment"], exp_cfg=EXP_CFG), indent=2), encoding="utf-8")
print("PROTOCOL_HASH", PROTOCOL_HASH)


## 6. Dataset (saved crop + saved 600 mask)


In [ ]:
IMAGENET_MEAN_NP = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD_NP = np.array([0.229, 0.224, 0.225], dtype=np.float32)
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def letterbox_rgb(image_rgb, size: int):
    h, w = image_rgb.shape[:2]
    scale = min(size / max(h, 1), size / max(w, 1))
    nw, nh = max(int(round(w * scale)), 1), max(int(round(h * scale)), 1)
    resized = cv2.resize(image_rgb, (nw, nh), interpolation=cv2.INTER_LINEAR)
    canvas = np.zeros((size, size, 3), dtype=np.uint8)
    left, top = (size - nw) // 2, (size - nh) // 2
    canvas[top: top + nh, left: left + nw] = resized
    return canvas


def letterbox_valid_mask(crop, size):
    h, w = crop.shape[:2]
    scale = min(size / h, size / w)
    nw, nh = max(round(w * scale), 1), max(round(h * scale), 1)
    left, top = (size - nw) // 2, (size - nh) // 2
    result = np.zeros((size, size), dtype=np.float32)
    result[top:top+nh, left:left+nw] = 1
    return result


class GeneratedMaskDataset(Dataset):
    def __init__(self, df: pd.DataFrame, size: int, augment: bool = False):
        self.df = df.reset_index(drop=True)
        self.size = int(size)
        self.augment = bool(augment)
        self.epoch = 0
        self.seed = int(CONFIG["seed"])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        rng = random.Random(self.seed + self.epoch * len(self.df) + int(idx))
        bgr = cv2.imread(row.path, cv2.IMREAD_COLOR)
        if bgr is None:
            raise FileNotFoundError(f"Unreadable training image: {row.path}")
        else:
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            h, w = rgb.shape[:2]
            x0 = int(float(row.crop_x0)); y0 = int(float(row.crop_y0))
            x1 = int(float(row.crop_x1)); y1 = int(float(row.crop_y1))
            x0, y0 = max(0, x0), max(0, y0)
            x1, y1 = min(w, max(x0 + 1, x1)), min(h, max(y0 + 1, y1))
        crop = rgb[y0:y1, x0:x1]
        mask = cv2.imread(str(row.mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise FileNotFoundError(f"Unreadable generated mask: {row.mask_path}")
        if mask.shape != (self.size, self.size):
            mask = cv2.resize(mask, (self.size, self.size), interpolation=cv2.INTER_NEAREST)
        if self.augment and rng.random() < 0.5:
            crop = np.clip(crop.astype(np.float32) * (1.0 + rng.uniform(-0.15, 0.15)) + rng.uniform(-12, 12), 0, 255).astype(np.uint8)
        canvas = letterbox_rgb(crop, self.size)
        valid_canvas = letterbox_valid_mask(crop, self.size)
        if self.augment and rng.random() < 0.5:
            canvas = np.ascontiguousarray(canvas[:, ::-1])
            mask = np.ascontiguousarray(mask[:, ::-1])
            valid_canvas = np.ascontiguousarray(valid_canvas[:, ::-1])
        x = canvas.astype(np.float32) / 255.0
        x = (x - IMAGENET_MEAN_NP) / IMAGENET_STD_NP
        x = np.transpose(x, (2, 0, 1))
        target = (mask.astype(np.float32) > 127).astype(np.float32)
        return {
            "image": torch.from_numpy(x.copy()),
            "mask": torch.from_numpy(target[None].copy()),
            "valid_mask": torch.from_numpy(valid_canvas[None].copy()),
            "seg_valid": torch.tensor(float(row.seg_valid)),
            "rel_key": row.rel_key,
            "label": torch.tensor([float(row.label)], dtype=torch.float32),
            "path": row.path,
            "file_name": row.file_name,
        }


class EpochSeededSampler(torch.utils.data.Sampler):
    def __init__(self, n, seed):
        self.n = int(n)
        self.seed = int(seed)
        self.epoch = 0
        self.start_index = 0

    def set_epoch(self, epoch, start_index=0):
        self.epoch = int(epoch)
        self.start_index = int(start_index)

    def __iter__(self):
        g = torch.Generator()
        g.manual_seed(self.seed * 1000 + self.epoch)
        order = torch.randperm(self.n, generator=g).tolist()
        return iter(order[self.start_index:])

    def __len__(self):
        return max(self.n - self.start_index, 0)


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_loader(df, shuffle, augment):
    ds = GeneratedMaskDataset(df, CONFIG["image_size"], augment=augment)
    sampler = EpochSeededSampler(len(ds), CONFIG["seed"]) if shuffle else None
    return DataLoader(
        ds,
        batch_size=CONFIG["batch_size"],
        sampler=sampler,
        shuffle=False,
        num_workers=CONFIG["num_workers"],
        worker_init_fn=seed_worker,
        generator=torch.Generator().manual_seed(CONFIG["seed"]),
        pin_memory=torch.cuda.is_available(),
        drop_last=bool(shuffle and len(ds) >= int(CONFIG["batch_size"])),
    )


if int(CONFIG["image_size"]) != 600:
    raise RuntimeError("Generated masks are 600×600. Leave image_size at 600.")
TRAIN_LOADER = make_loader(INDEX["train"], True, True)
VAL_LOADER = make_loader(INDEX["val"], False, False)
TEST_LOADER = make_loader(INDEX["test"], False, False)
print("train batches", len(TRAIN_LOADER), "val", len(VAL_LOADER), "test", len(TEST_LOADER))


## 7. Model


In [ ]:
def _srm_kernels() -> torch.Tensor:
    k1 = np.array(
        [[0, 0, 0, 0, 0], [0, -1, 2, -1, 0], [0, 2, -4, 2, 0], [0, -1, 2, -1, 0], [0, 0, 0, 0, 0]],
        dtype=np.float32,
    )
    k2 = np.array(
        [[-1, 2, -2, 2, -1], [2, -6, 8, -6, 2], [-2, 8, -12, 8, -2], [2, -6, 8, -6, 2], [-1, 2, -2, 2, -1]],
        dtype=np.float32,
    )
    k3 = np.array(
        [[0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 1, -2, 1, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]],
        dtype=np.float32,
    )
    ks = []
    for k in (k1, k2, k3):
        s = np.abs(k).sum()
        ks.append(k / (s if s > 0 else 1.0))
    return torch.from_numpy(np.stack(ks, 0)[:, None, :, :])

class ASPP(nn.Module):
    def __init__(self, in_ch: int, out_ch: int = 256, rates=(6, 12, 18)):
        super().__init__()
        self.branches = nn.ModuleList()
        self.branches.append(nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)))
        for r in rates:
            self.branches.append(nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=r, dilation=r, bias=False),
                nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            ))
        gn = 32 if out_ch % 32 == 0 else 1
        self.gap = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.GroupNorm(gn, out_ch), nn.ReLU(inplace=True),
        )
        self.project = nn.Sequential(
            nn.Conv2d(out_ch * 5, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True), nn.Dropout(0.1),
        )

    def forward(self, x):
        res = [b(x) for b in self.branches]
        gap = F.interpolate(self.gap(x), size=x.shape[-2:], mode="bilinear", align_corners=False)
        res.append(gap)
        return self.project(torch.cat(res, dim=1))

class DeepLabV3PlusHead(nn.Module):
    def __init__(self, high_ch: int, low_ch: int, num_classes: int = 1):
        super().__init__()
        self.aspp = ASPP(high_ch, 256)
        self.low_proj = nn.Sequential(nn.Conv2d(low_ch, 48, 1, bias=False), nn.BatchNorm2d(48), nn.ReLU(inplace=True))
        self.decoder = nn.Sequential(
            nn.Conv2d(304, 256, 3, padding=1, bias=False), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1, bias=False), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, num_classes, 1),
        )

    def forward(self, high, low, out_size):
        high = F.interpolate(self.aspp(high), size=low.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([high, self.low_proj(low)], dim=1)
        return F.interpolate(self.decoder(x), size=out_size, mode="bilinear", align_corners=False)

class ClassificationHead(nn.Module):
    def __init__(self, in_ch: int, dropout: float = 0.5):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_ch, 1))

    def forward(self, feat):
        return self.fc(self.pool(feat).flatten(1))

"""Versioned OW models: corrected early fusion and an experimental late fusion.

Legacy checkpoints continue to use architecture.py. These models require fresh
training and record implementation_version='ow_v2' in their checkpoints.
"""

import torch

from torch import nn

from torch.nn import functional as F

from torchvision.models import EfficientNet_B7_Weights, efficientnet_b7

IMPLEMENTATION_VERSION = "ow_v2"

class ForensicStreams(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer("srm_weight", _srm_kernels())

    def forward(self, normalized_rgb):
        # FFT on 600px images must stay float32 even under CUDA autocast.
        with torch.autocast(device_type=normalized_rgb.device.type, enabled=False):
            x = normalized_rgb.float()
            mean = IMAGENET_MEAN.to(x.device)
            std = IMAGENET_STD.to(x.device)
            rgb = (x * std + mean).clamp(0, 1)
            gray = (rgb * x.new_tensor([0.299, 0.587, 0.114]).view(1, 3, 1, 1)).sum(1, keepdim=True)
            residual = F.conv2d(gray, self.srm_weight.float(), padding=2).tanh()
            spectrum = torch.fft.fftshift(torch.fft.fft2(rgb, norm="ortho"), dim=(-2, -1))
            magnitude = torch.log1p(spectrum.abs())
            lo = magnitude.amin(dim=(-2, -1), keepdim=True)
            span = magnitude.amax(dim=(-2, -1), keepdim=True) - lo
            magnitude = (magnitude - lo) / span.clamp_min(1e-6)
            magnitude = magnitude - magnitude.mean(dim=(-2, -1), keepdim=True)
        return residual, magnitude

def expand_rgb_stem(conv3):
    """Preserve the RGB response exactly; learn the auxiliary channels from zero."""
    conv9 = nn.Conv2d(9, conv3.out_channels, conv3.kernel_size, stride=conv3.stride,
                      padding=conv3.padding, dilation=conv3.dilation,
                      bias=conv3.bias is not None, device=conv3.weight.device, dtype=conv3.weight.dtype)
    with torch.no_grad():
        conv9.weight.zero_()
        conv9.weight[:, :3].copy_(conv3.weight)
        if conv3.bias is not None:
            conv9.bias.copy_(conv3.bias)
    return conv9

class AuxiliaryEncoder(nn.Sequential):
    def __init__(self):
        layers = []
        channels = 3
        for width in (32, 64, 128):
            layers.extend([nn.Conv2d(channels, width, 3, stride=2, padding=1, bias=False),
                           nn.GroupNorm(8, width), nn.SiLU()])
            channels = width
        layers.extend([nn.AdaptiveAvgPool2d(1), nn.Flatten(1)])
        super().__init__(*layers)

class VERITASV2(nn.Module):
    def __init__(self, multi_stream=True, segmentation=True, pretrained=True, fusion="early"):
        super().__init__()
        if fusion not in ("rgb", "early", "late") or multi_stream != (fusion != "rgb"):
            raise ValueError("Use fusion='rgb' for RGB-only models, otherwise 'early' or 'late'")
        self.multi_stream = bool(multi_stream)
        self.segmentation = bool(segmentation)
        self.fusion = fusion
        weights = EfficientNet_B7_Weights.IMAGENET1K_V1 if pretrained else None
        self.encoder = efficientnet_b7(weights=weights).features
        # Create the common RGB backbone/head before variant-specific parameters.
        self.cls_head = ClassificationHead(2560)
        self.seg_head = DeepLabV3PlusHead(224, 48, 1) if segmentation else None
        self.streams = ForensicStreams() if multi_stream else None
        if fusion == "early":
            self.encoder[0][0] = expand_rgb_stem(self.encoder[0][0])
        elif fusion == "late":
            self.srm_encoder = AuxiliaryEncoder()
            self.fft_encoder = AuxiliaryEncoder()
            self.aux_projection = nn.Linear(256, 2560, bias=False)
            nn.init.zeros_(self.aux_projection.weight)

    def forward(self, x):
        size = x.shape[-2:]
        residual = magnitude = None
        if self.multi_stream:
            residual, magnitude = self.streams(x)
        if self.fusion == "early":
            # Keep the original ImageNet-normalized RGB, unlike the legacy stem.
            x = torch.cat([x, residual, magnitude], dim=1)
        low = high = None
        for i, stage in enumerate(self.encoder):
            x = stage(x)
            if i == 2:
                low = x
            elif i == 5:
                high = x
        if self.fusion == "late":
            # Pool independent domains before fusion: frequency coordinates are
            # never treated as image locations by the segmentation decoder.
            auxiliary = torch.cat([self.srm_encoder(residual), self.fft_encoder(magnitude)], dim=1)
            pooled = self.cls_head.pool(x).flatten(1) + self.aux_projection(auxiliary)
            logits = self.cls_head.fc(pooled)
        else:
            logits = self.cls_head(x)
        out = {"classification_logits": logits, "fake_probability": torch.sigmoid(logits)}
        if self.segmentation:
            seg_logits = self.seg_head(high, low, out_size=size)
            out.update(segmentation_logits=seg_logits, segmentation_prob=torch.sigmoid(seg_logits))
        return out

"""OW v3 candidates and references, with an unchanged direct RGB B7 path.

Local FFT callers should pass a binary rectangular ``valid_mask`` (N,1,H,W)
covering the letterboxed content, including its exact offset after augmentation.
Absent masks mean the entire tensor is content; black pixels never imply padding.
This is a model candidate, not a claim of improved trained accuracy.
"""

import math

import torch

from torch import nn

from torch.nn import functional as F

from torchvision.models import EfficientNet_B7_Weights, efficientnet_b7

IMPLEMENTATION_VERSION = "ow_v3"

class ForensicStreamsV3(nn.Module):
    """Color residuals and optional padding-aware local spectral statistics.

Local FFT uses nonoverlapping 32px patches, patch-mean removal, a Hann taper,
and three radial bands [0,.125), [.125,.25), [.25,sqrt(.5)] cycles/pixel.
Replicate extension fills incomplete edge patches; no zero letterbox enters FFT.
Band powers retain their amplitude via log1p, without per-image min/max scaling.
"""
    def __init__(self, color_srm=False, frequency="global"):
        super().__init__()
        if frequency not in ("global", "local"):
            raise ValueError("frequency must be 'global' or 'local'")
        self.color_srm = bool(color_srm)
        self.frequency = frequency
        self.register_buffer("srm_weight", _srm_kernels())
        self.patch_size = 32
        self.grid_size = 32
        window = torch.hann_window(self.patch_size, periodic=False)
        self.register_buffer("fft_window", window[:, None] * window[None, :])
        fy = torch.fft.fftfreq(self.patch_size)[:, None]
        fx = torch.fft.rfftfreq(self.patch_size)[None, :]
        radius = torch.sqrt(fy.square() + fx.square())
        bands = torch.stack([(radius < 0.125), ((radius >= 0.125) & (radius < 0.25)), (radius >= 0.25)]).float()
        self.register_buffer("fft_bands", bands / bands.sum((-2, -1), keepdim=True))

    @staticmethod
    def content_bounds(rgb, valid_mask):
        if valid_mask is None:
            return [(0, rgb.shape[-2], 0, rgb.shape[-1])] * rgb.shape[0]
        if valid_mask.shape != (rgb.shape[0], 1, *rgb.shape[-2:]):
            raise ValueError("valid_mask must have shape (N,1,H,W) matching RGB")
        if not torch.isfinite(valid_mask).all():
            raise ValueError("valid_mask must be finite and binary")
        if not ((valid_mask == 0) | (valid_mask == 1)).all():
            raise ValueError("valid_mask must be binary")
        result = []
        for mask in valid_mask[:, 0]:
            positions = torch.nonzero(mask, as_tuple=False)
            if positions.numel() == 0:
                raise ValueError("valid_mask must contain nonempty rectangular content")
            y0, x0 = positions.amin(0).tolist()
            y1, x1 = (positions.amax(0) + 1).tolist()
            if int(mask.sum()) != (y1 - y0) * (x1 - x0):
                raise ValueError("valid_mask must describe one filled rectangle")
            result.append((y0, y1, x0, x1))
        return result

    def local_frequency(self, rgb, valid_mask):
        outputs = []
        patch = self.patch_size
        for sample, (y0, y1, x0, x1) in zip(rgb, self.content_bounds(rgb, valid_mask)):
            crop = sample[None, :, y0:y1, x0:x1]
            height, width = crop.shape[-2:]
            crop = F.pad(crop, (0, (-width) % patch, 0, (-height) % patch), mode="replicate")
            patches = crop.unfold(2, patch, patch).unfold(3, patch, patch)
            patches = patches - patches.mean((-2, -1), keepdim=True)
            power = torch.fft.rfft2(patches * self.fft_window, norm="ortho").abs().square()
            band_power = torch.einsum("bcijuv,kuv->bckij", power, self.fft_bands)
            stats = torch.log1p(band_power).flatten(1, 2)
            outputs.append(F.adaptive_avg_pool2d(stats, (self.grid_size, self.grid_size)))
        return torch.cat(outputs, dim=0)

    def forward(self, normalized_rgb, valid_mask=None):
        # CUDA FFT on 600px inputs and all spectral reductions stay float32.
        with torch.autocast(device_type=normalized_rgb.device.type, enabled=False):
            x = normalized_rgb.float()
            rgb = (x * IMAGENET_STD.to(x.device) + IMAGENET_MEAN.to(x.device)).clamp(0, 1)
            if self.color_srm:
                residual = F.conv2d(rgb, self.srm_weight.float().repeat(3, 1, 1, 1), padding=2, groups=3).tanh()
            else:
                gray = (rgb * x.new_tensor([0.299, 0.587, 0.114]).view(1, 3, 1, 1)).sum(1, keepdim=True)
                residual = F.conv2d(gray, self.srm_weight.float(), padding=2).tanh()
            if self.frequency == "local":
                magnitude = self.local_frequency(rgb, valid_mask)
            else:
                spectrum = torch.fft.fftshift(torch.fft.fft2(rgb, norm="ortho"), dim=(-2, -1))
                magnitude = torch.log1p(spectrum.abs())
                lo = magnitude.amin((-2, -1), keepdim=True)
                span = magnitude.amax((-2, -1), keepdim=True) - lo
                magnitude = (magnitude - lo) / span.clamp_min(1e-6)
                magnitude = magnitude - magnitude.mean((-2, -1), keepdim=True)
        return residual, magnitude

class EvidencePoolV3(nn.Module):
    def __init__(self, mode="avg"):
        super().__init__()
        if mode not in ("avg", "avg_max"):
            raise ValueError("pooling must be 'avg' or 'avg_max'")
        self.mode = mode

    def forward(self, feature):
        average = F.adaptive_avg_pool2d(feature, 1).flatten(1)
        if self.mode == "avg":
            return average
        maximum = F.adaptive_max_pool2d(feature, 1).flatten(1)
        return torch.cat([average, maximum], dim=1)

class AuxiliaryEncoderV3(nn.Sequential):
    def __init__(self, channels=3, pooling="avg"):
        layers = []
        for width in (32, 64, 128):
            layers.extend([nn.Conv2d(channels, width, 3, stride=2, padding=1, bias=False),
                           nn.GroupNorm(8, width), nn.SiLU()])
            channels = width
        layers.append(EvidencePoolV3(pooling))
        super().__init__(*layers)

class ResidualFusionV3(nn.Sequential):
    """Only the final layer starts at zero; hidden features can learn after step one."""
    def __init__(self, channels, hidden=256):
        super().__init__(nn.LayerNorm(channels), nn.Linear(channels, hidden), nn.SiLU(),
                         nn.Dropout(0.2), nn.Linear(hidden, 1))
        nn.init.zeros_(self[-1].weight)
        nn.init.zeros_(self[-1].bias)

def _gn_v3(channels):
    return nn.GroupNorm(math.gcd(32, channels), channels)

class SeparableConvV3(nn.Sequential):
    def __init__(self, in_channels, out_channels, dilation=1):
        super().__init__(
            nn.Conv2d(in_channels, in_channels, 3, padding=dilation, dilation=dilation,
                      groups=in_channels, bias=False),
            nn.Conv2d(in_channels, out_channels, 1, bias=False),
            _gn_v3(out_channels), nn.ReLU(inplace=True))

class SeparableASPPV3(nn.Module):
    def __init__(self, in_channels=224, channels=256, rates=(6, 12, 18)):
        super().__init__()
        self.branches = nn.ModuleList([nn.Sequential(nn.Conv2d(in_channels, channels, 1, bias=False),
                                                    _gn_v3(channels), nn.ReLU(inplace=True))])
        self.branches.extend(SeparableConvV3(in_channels, channels, rate) for rate in rates)
        self.gap = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(in_channels, channels, 1, bias=False),
                                 _gn_v3(channels), nn.ReLU(inplace=True))
        self.project = nn.Sequential(nn.Conv2d(channels * 5, channels, 1, bias=False),
                                     _gn_v3(channels), nn.ReLU(inplace=True), nn.Dropout(0.1))

    def forward(self, x):
        features = [branch(x) for branch in self.branches]
        features.append(F.interpolate(self.gap(x), size=x.shape[-2:], mode="bilinear", align_corners=False))
        return self.project(torch.cat(features, dim=1))

class SeparableDeepLabHeadV3(nn.Module):
    """The two 150px decoder blocks cost 3.339G rather than 29.0304G MAC/image.

    These operation counts apply at 600px input and unchanged 304/256 widths,
    and do not predict the whole model's wall-clock speedup or trained accuracy.
    """
    def __init__(self, high_channels=224, low_channels=48):
        super().__init__()
        self.aspp = SeparableASPPV3(high_channels)
        self.low_proj = nn.Sequential(nn.Conv2d(low_channels, 48, 1, bias=False),
                                      _gn_v3(48), nn.ReLU(inplace=True))
        self.decoder = nn.Sequential(SeparableConvV3(304, 256), SeparableConvV3(256, 256), nn.Conv2d(256, 1, 1))

    def forward(self, high, low, out_size):
        high = F.interpolate(self.aspp(high), size=low.shape[-2:], mode="bilinear", align_corners=False)
        logits = self.decoder(torch.cat([high, self.low_proj(low)], dim=1))
        return F.interpolate(logits, size=out_size, mode="bilinear", align_corners=False)

class VERITASV3(nn.Module):
    def __init__(self, multi_stream=False, segmentation=False, pretrained=True, fusion="rgb",
                 color_srm=False, frequency="global", pooling="avg", decoder="dense"):
        super().__init__()
        if fusion not in ("rgb", "early", "late_additive", "late_nonlinear") or bool(multi_stream) != (fusion != "rgb"):
            raise ValueError("RGB-only requires fusion='rgb'; multistream requires an early/late fusion mode")
        if frequency not in ("global", "local") or pooling not in ("avg", "avg_max") or decoder not in ("dense", "separable"):
            raise ValueError("Unsupported frequency, pooling, or decoder mode")
        if fusion == "early" and (color_srm or frequency != "global"):
            raise ValueError("The corrected early reference retains grayscale SRM/global FFT and 9 input channels")
        self.multi_stream = bool(multi_stream)
        self.segmentation = bool(segmentation)
        self.fusion, self.pooling = fusion, pooling
        self.color_srm, self.frequency, self.decoder = bool(color_srm), frequency, decoder
        self.requires_valid_mask = self.multi_stream and frequency == "local"
        weights = EfficientNet_B7_Weights.IMAGENET1K_V1 if pretrained else None
        self.encoder = efficientnet_b7(weights=weights).features
        self.cls_head = ClassificationHead(2560)
        if self.segmentation:
            self.seg_head = DeepLabV3PlusHead(224, 48, 1) if decoder == "dense" else SeparableDeepLabHeadV3()
        else:
            self.seg_head = None
        self.streams = ForensicStreamsV3(color_srm, frequency) if self.multi_stream else None
        self.evidence_pool = EvidencePoolV3(pooling)
        multiplier = 2 if pooling == "avg_max" else 1
        if fusion == "early":
            self.encoder[0][0] = expand_rgb_stem(self.encoder[0][0])
        elif fusion.startswith("late"):
            self.srm_encoder = AuxiliaryEncoderV3(9 if color_srm else 3, pooling)
            self.fft_encoder = AuxiliaryEncoderV3(9 if frequency == "local" else 3, pooling)
            if fusion == "late_additive":
                self.aux_projection = nn.Linear(256 * multiplier, 2560, bias=False)
                nn.init.zeros_(self.aux_projection.weight)
            else:
                self.fusion_head = ResidualFusionV3((2560 + 256) * multiplier)
        if pooling == "avg_max" and fusion != "late_nonlinear":
            self.local_head = ResidualFusionV3(2560 * 2)

    def forward(self, x, valid_mask=None):
        size = x.shape[-2:]
        residual = magnitude = None
        if self.multi_stream:
            residual, magnitude = self.streams(x, valid_mask)
        if self.fusion == "early":
            x = torch.cat([x, residual, magnitude], dim=1)
        low = high = None
        for i, stage in enumerate(self.encoder):
            x = stage(x)
            if i == 2:
                low = x
            elif i == 5:
                high = x
        rgb_average = self.cls_head.pool(x).flatten(1)
        rgb_evidence = self.evidence_pool(x)
        if self.fusion == "late_additive":
            auxiliary = torch.cat([self.srm_encoder(residual), self.fft_encoder(magnitude)], dim=1)
            logits = self.cls_head.fc(rgb_average + self.aux_projection(auxiliary))
        else:
            logits = self.cls_head.fc(rgb_average)
            if self.fusion == "late_nonlinear":
                auxiliary = torch.cat([self.srm_encoder(residual), self.fft_encoder(magnitude)], dim=1)
                logits = logits + self.fusion_head(torch.cat([rgb_evidence, auxiliary], dim=1))
        if hasattr(self, "local_head"):
            logits = logits + self.local_head(rgb_evidence)
        out = {"classification_logits": logits, "fake_probability": logits.sigmoid()}
        if self.segmentation:
            seg_logits = self.seg_head(high, low, out_size=size)
            out.update(segmentation_logits=seg_logits, segmentation_prob=seg_logits.sigmoid())
        return out

def build_model():
    args = {k: v for k, v in EXP_CFG.items() if k != "implementation_version"}
    return VERITASV3(pretrained=CONFIG["use_pretrained"], **args).to(DEVICE)

MODEL_SOURCE_HASH = "3ccf94aa98c9313ed164441883dd1c3f3fafbf2044a3eea8626daf778eb23dce"


In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])


## 8. Loss


In [ ]:
"""OW v3 supervision and optimization helpers, embedded into Kaggle notebooks.

Real class labels establish zero manipulation targets. Fake localization remains
teacher supervision, not independent ground truth or calibrated confidence.
"""
import math

import torch
from torch import nn
from torch.nn import functional as F


def _binary_tensor(value, name):
    if not torch.is_tensor(value):
        raise TypeError(f"{name} must be a tensor")
    if not torch.isfinite(value).all() or ((value != 0) & (value != 1)).any():
        raise ValueError(f"{name} must contain only finite binary values 0 and 1")


class VERITASLossV3(nn.Module):
    """Class BCE plus label-consistent, empty-aware auxiliary segmentation.

    BCE is averaged over valid canvas pixels separately for each trusted image.
    A Real image contributes its full BCE. A trusted nonempty Fake contributes
    (BCE + dice_weight * foreground_Dice) / (1 + dice_weight). Those per-image
    losses are averaged, keeping each trusted image's total weight equal.

    ``seg_valid`` can exclude unreliable Fake masks; it never overrides known
    Real negatives. Mask area determines target availability, not confidence.
    Empty Fake targets are unavailable localization supervision and are skipped.
    """

    def __init__(self, lambda_cls=1.0, lambda_seg=0.25, dice_weight=1.0, segmentation=False):
        super().__init__()
        values = (float(lambda_cls), float(lambda_seg), float(dice_weight))
        if not all(math.isfinite(value) for value in values) or values[0] <= 0 or min(values[1:]) < 0:
            raise ValueError("Loss weights must be finite and nonnegative, with positive classification weight")
        self.lambda_cls, self.lambda_seg, self.dice_weight = values
        self.segmentation = bool(segmentation)

    def forward(self, outputs, labels, masks=None, valid_mask=None, seg_valid=None):
        cls_logits = outputs["classification_logits"]
        if cls_logits.ndim != 2 or cls_logits.shape[1] != 1 or cls_logits.shape[0] == 0:
            raise ValueError("Classification logits must have nonempty shape [N, 1]")
        _binary_tensor(labels, "labels")
        if labels.shape != cls_logits.shape:
            raise ValueError("Labels must have the same [N, 1] shape as classification logits")
        labels = labels.to(device=cls_logits.device, dtype=torch.float32)
        cls = F.binary_cross_entropy_with_logits(cls_logits.float(), labels)
        seg = seg_bce = seg_dice = cls.new_zeros(())
        counts = {key: cls.new_zeros(()) for key in (
            "segmentation_valid_samples", "segmentation_positive_samples",
            "segmentation_ignored_fake_samples", "segmentation_valid_pixels",
        )}
        if self.segmentation:
            if masks is None:
                raise ValueError("Segmentation targets are required")
            logits = outputs["segmentation_logits"]
            if logits.ndim != 4 or logits.shape[:2] != (len(labels), 1):
                raise ValueError("Segmentation logits must have shape [N, 1, H, W]")
            if masks.shape != logits.shape:
                raise ValueError("Masks must have the same shape as segmentation logits")
            _binary_tensor(masks, "masks")
            targets = masks.to(device=logits.device, dtype=torch.float32)
            real = labels[:, 0].to(logits.device) == 0
            # Preserve the original teacher PNG; correct targets only in memory.
            targets = torch.where(real[:, None, None, None], torch.zeros_like(targets), targets)
            if valid_mask is None:
                valid = torch.ones_like(targets)
            else:
                if valid_mask.shape != targets.shape:
                    raise ValueError("valid_mask must have the same shape as segmentation targets")
                _binary_tensor(valid_mask, "valid_mask")
                valid = valid_mask.to(device=logits.device, dtype=torch.float32)
            if seg_valid is None:
                reliability = torch.ones(len(labels), dtype=torch.bool, device=logits.device)
            else:
                if seg_valid.shape not in ((len(labels),), (len(labels), 1)):
                    raise ValueError("seg_valid must have shape [N] or [N, 1]")
                _binary_tensor(seg_valid, "seg_valid")
                reliability = seg_valid.reshape(-1).to(device=logits.device, dtype=torch.bool)
            dims = (1, 2, 3)
            pixel_counts = valid.sum(dims)
            nonempty = (targets * valid).sum(dims) > 0
            trusted = (pixel_counts > 0) & (real | (reliability & nonempty))
            positive = trusted & ~real & nonempty
            trusted_weight, positive_weight = trusted.float(), positive.float()
            # Float32 also protects 600px reductions inside CUDA autocast.
            seg_logits = logits.float()
            per_bce = (F.binary_cross_entropy_with_logits(seg_logits, targets, reduction="none") * valid).sum(dims)
            per_bce = per_bce / pixel_counts.clamp_min(1)
            probs = seg_logits.sigmoid() * valid
            foreground = targets * valid
            per_dice = 1 - ((2 * (probs * foreground).sum(dims) + 1e-6) /
                            (probs.sum(dims) + foreground.sum(dims) + 1e-6))
            per_image = torch.where(positive, (per_bce + self.dice_weight * per_dice) /
                                    (1 + self.dice_weight), per_bce)
            # Weighted sums remain differentiable zero when no target is trusted.
            seg = (per_image * trusted_weight).sum() / trusted_weight.sum().clamp_min(1)
            seg_bce = (per_bce * trusted_weight).sum() / trusted_weight.sum().clamp_min(1)
            seg_dice = (per_dice * positive_weight).sum() / positive_weight.sum().clamp_min(1)
            counts = dict(segmentation_valid_samples=trusted_weight.sum(),
                          segmentation_positive_samples=positive_weight.sum(),
                          segmentation_ignored_fake_samples=(~real & ~trusted).float().sum(),
                          segmentation_valid_pixels=(pixel_counts * trusted_weight).sum())
        total = self.lambda_cls * cls + self.lambda_seg * seg
        return dict(total=total, classification=cls.detach(), segmentation=seg.detach(),
                    segmentation_bce=seg_bce.detach(), segmentation_dice=seg_dice.detach(),
                    **{key: value.detach() for key, value in counts.items()})


def validate_resume_checkpoint(checkpoint, experiment, exp_cfg, protocol_hash):
    if checkpoint.get("experiment_name") != experiment:
        raise ValueError("Checkpoint belongs to a different experiment")
    if checkpoint.get("exp_cfg") != exp_cfg:
        raise ValueError("Checkpoint architecture/version differs; start a fresh experiment")
    if checkpoint.get("protocol_hash") != protocol_hash:
        raise ValueError("Checkpoint protocol differs (including smoke/full or mask source)")


def build_optimizer(model, learning_rate, weight_decay, backbone_lr_multiplier=0.1):
    """Use a smaller LR for pretrained encoder parameters, with exact coverage."""
    learning_rate, weight_decay, multiplier = map(float, (learning_rate, weight_decay, backbone_lr_multiplier))
    if not all(math.isfinite(value) for value in (learning_rate, weight_decay, multiplier)):
        raise ValueError("Optimizer settings must be finite")
    if learning_rate <= 0 or weight_decay < 0 or multiplier <= 0:
        raise ValueError("Learning rates must be positive and weight decay nonnegative")
    if not isinstance(getattr(model, "encoder", None), nn.Module):
        raise ValueError("Model must expose its pretrained encoder as model.encoder")
    backbone = [p for p in model.encoder.parameters() if p.requires_grad]
    backbone_ids = {id(p) for p in backbone}
    heads = [p for p in model.parameters() if p.requires_grad and id(p) not in backbone_ids]
    groups = []
    if backbone:
        groups.append(dict(params=backbone, lr=learning_rate * multiplier, name="backbone"))
    if heads:
        groups.append(dict(params=heads, lr=learning_rate, name="heads"))
    if not groups:
        raise ValueError("Model has no trainable parameters")
    return torch.optim.AdamW(groups, lr=learning_rate, weight_decay=weight_decay)


def build_scheduler(optimizer, total_updates, warmup_fraction=0.05):
    """Step after each successful optimizer update; save/load its state on resume."""
    if isinstance(total_updates, bool) or int(total_updates) != total_updates or total_updates <= 0:
        raise ValueError("total_updates must be a positive integer")
    total_updates = int(total_updates)
    fraction = float(warmup_fraction)
    if not math.isfinite(fraction) or not 0 <= fraction < 1:
        raise ValueError("warmup_fraction must be finite and in [0, 1)")
    warmup_updates = min(max(int(round(total_updates * fraction)), 1), total_updates) if fraction else 0

    def factor(update):
        if update < warmup_updates:
            return (update + 1) / warmup_updates
        progress = min(max((update - warmup_updates) / max(total_updates - warmup_updates, 1), 0), 1)
        return 0.5 * (1 + math.cos(math.pi * progress))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, factor)


def freeze_backbone_bn(model):
    """Freeze encoder BN statistics after model.train(); keep affine gradients."""
    if not isinstance(getattr(model, "encoder", None), nn.Module):
        raise ValueError("Model must expose its pretrained encoder as model.encoder")
    for module in model.encoder.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()


## 9. Checkpoint resume (one epoch per run)


In [ ]:
# Checkpoint and training functions are defined together below.


## 10. Train and test


In [ ]:
"""Training/evaluation runtime embedded verbatim in the self-contained OW v3 notebooks.

Notebook setup supplies CONFIG, EXP_CFG, INDEX, WORKING, loaders, DEVICE,
PROTOCOL_HASH, MODEL_SOURCE_HASH, EpochSeededSampler, set_seed and build_model.
Importing this module does not start training.
"""
import gc
import hashlib
import json
import math
from pathlib import Path
import shutil
import time
import zipfile

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from tqdm.auto import tqdm


def torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def stage_checkpoints():
    if not CONFIG.get("resume", True) or not CONFIG.get("resume_from_input", True):
        return
    root = Path("/kaggle/input")
    if not root.exists():
        return
    for suffix in ("last", "best"):
        name = f'{CONFIG["experiment"]}_{suffix}.pt'
        dst = WORKING / "checkpoints" / name
        candidates = list(root.rglob(name)) + ([dst] if dst.exists() else [])
        extracted = []
        # Kaggle can wrap outputs in a ZIP; extract only this experiment's files.
        for archive in root.rglob("_output_.zip"):
            with zipfile.ZipFile(archive) as z:
                for member in z.namelist():
                    if Path(member.replace("\\", "/")).name == name:
                        identity = hashlib.sha256((str(archive.resolve()) + ":" + member).encode()).hexdigest()[:20]
                        staged = WORKING / "checkpoints" / ("_stage_" + identity + "_" + name)
                        with z.open(member) as source, staged.open("wb") as target:
                            shutil.copyfileobj(source, target)
                        candidates.append(staged)
                        extracted.append(staged)
        selected, progress = None, (-1, -1)
        for path in candidates:
            ck = torch_load(path)
            validate_resume_checkpoint(ck, CONFIG["experiment"], EXP_CFG, PROTOCOL_HASH)
            score = (int(ck["epoch"]), int((ck.get("partial") or {}).get("step", 0)))
            if score > progress:
                selected, progress = path, score
            del ck
        if selected is not None and selected != dst:
            shutil.copy2(selected, dst)
            print("STAGED", name, "progress", progress)
        for path in extracted:
            path.unlink()


def load_checkpoint(path, model, optimizer=None, scaler=None, scheduler=None):
    ck = torch_load(path)
    validate_resume_checkpoint(ck, CONFIG["experiment"], EXP_CFG, PROTOCOL_HASH)
    model.load_state_dict(ck["model_state_dict"], strict=True)
    if optimizer is not None:
        optimizer.load_state_dict(ck["optimizer_state_dict"])
    if scaler is not None and ck.get("scaler_state_dict"):
        scaler.load_state_dict(ck["scaler_state_dict"])
    if scheduler is not None:
        scheduler.load_state_dict(ck["scheduler_state_dict"])
    return ck


def save_checkpoint(path, model, optimizer, scaler, scheduler, epoch, best_metric, history, partial=None):
    payload = dict(epoch=epoch, partial=partial, best_metric=best_metric, history=history,
                   model_state_dict=model.state_dict(), optimizer_state_dict=optimizer.state_dict(),
                   scaler_state_dict=scaler.state_dict(), scheduler_state_dict=scheduler.state_dict(),
                   torch_rng_state=torch.get_rng_state(),
                   cuda_rng_state=torch.cuda.get_rng_state() if DEVICE.type == "cuda" else None,
                   experiment_name=CONFIG["experiment"], config=CONFIG, exp_cfg=EXP_CFG,
                   protocol_hash=PROTOCOL_HASH, model_source_hash=MODEL_SOURCE_HASH,
                   selection_metric="val_accuracy", supervision_policy="real_zero_fake_nonempty_v3")
    temporary = path.with_suffix(".tmp")
    torch.save(payload, temporary)
    temporary.replace(path)


@torch.no_grad()
def evaluate_loader(model, loader, split):
    model.eval()
    true, predicted, records, dice, iou, real_area = [], [], [], [], [], []
    for batch in loader:
        images = batch["image"].to(DEVICE, non_blocking=True)
        valid = batch["valid_mask"].to(DEVICE, non_blocking=True)
        outputs = model(images, valid_mask=valid)
        probabilities = outputs["fake_probability"].float().cpu().numpy().reshape(-1)
        labels = batch["label"].numpy().reshape(-1).astype(int)
        guesses = (probabilities >= CONFIG["classification_threshold"]).astype(int)
        true.extend(labels.tolist())
        predicted.extend(guesses.tolist())
        seg = outputs.get("segmentation_prob")
        for j, label in enumerate(labels):
            row = dict(rel_key=batch["rel_key"][j], file_name=batch["file_name"][j], split=split,
                       ground_truth_label=int(label), predicted_label=int(guesses[j]),
                       fake_probability=float(probabilities[j]))
            if seg is not None:
                content = valid[j, 0].bool()
                p = seg[j, 0][content] >= CONFIG["mask_threshold"]
                target = batch["mask"][j, 0].to(DEVICE)[content] > 0.5
                if label == 0:
                    area = float(p.float().mean())
                    real_area.append(area)
                    row["real_positive_fraction"] = area
                elif bool(batch["seg_valid"][j]) and bool(target.any()):
                    inter = float((p & target).sum())
                    denom, union = float(p.sum() + target.sum()), float((p | target).sum())
                    di, io = 2 * inter / max(denom, 1), inter / max(union, 1)
                    dice.append(di)
                    iou.append(io)
                    row.update(dice_teacher_fake=di, iou_teacher_fake=io)
            records.append(row)
    if not true:
        raise RuntimeError(f"No samples available for {split} evaluation")
    tn, fp, fn, tp = confusion_matrix(true, predicted, labels=[0, 1]).ravel()
    metrics = dict(accuracy=float(accuracy_score(true, predicted)),
                   f1=float(f1_score(true, predicted, zero_division=0)),
                   precision=float(precision_score(true, predicted, zero_division=0)),
                   recall=float(recall_score(true, predicted, zero_division=0)), n=len(true),
                   tp=int(tp), tn=int(tn), fp=int(fp), fn=int(fn))
    if dice:
        metrics.update(dice_teacher_fake_only=float(np.mean(dice)),
                       miou_teacher_fake_only=float(np.mean(iou)), n_seg_teacher_fake=len(dice))
    if real_area:
        metrics.update(real_mean_positive_fraction=float(np.mean(real_area)),
                       real_mask_nonempty_rate=float(np.mean(np.asarray(real_area) > 0)))
    return metrics, records


def train_one_experiment():
    set_seed(CONFIG["seed"])
    model = build_model()
    criterion = VERITASLossV3(CONFIG["lambda_cls"], CONFIG["lambda_seg"],
                             CONFIG["dice_weight"], EXP_CFG["segmentation"])
    optimizer = build_optimizer(model, CONFIG["learning_rate"], CONFIG["weight_decay"],
                                CONFIG["backbone_lr_multiplier"])
    accum = int(CONFIG["gradient_accumulation"])
    full_batches = len(TRAIN_LOADER)
    scheduler = build_scheduler(optimizer, math.ceil(full_batches / accum) * CONFIG["epochs"],
                                CONFIG["warmup_fraction"])
    amp = bool(CONFIG["use_amp"] and DEVICE.type == "cuda")
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=amp)
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler(enabled=amp)
    best_path = WORKING / "checkpoints" / f'{CONFIG["experiment"]}_best.pt'
    last_path = WORKING / "checkpoints" / f'{CONFIG["experiment"]}_last.pt'
    stage_checkpoints()
    resumed = {}
    if CONFIG.get("resume", True):
        resume_path = last_path if last_path.exists() else best_path
        if resume_path.exists():
            resumed = load_checkpoint(resume_path, model, optimizer, scaler, scheduler)
    elif last_path.exists() or best_path.exists():
        raise RuntimeError("Fresh training requires a new working session; existing checkpoints are present")
    completed = int(resumed.get("epoch", 0))
    best = float(resumed.get("best_metric", -1))
    history = list(resumed.get("history", []))
    partial = resumed.get("partial")
    print("START OF RUN", CONFIG["experiment"], completed, "/", CONFIG["epochs"])
    if CONFIG.get("skip_training", False):
        if completed != CONFIG["epochs"] or partial:
            raise RuntimeError("Evaluation-only mode requires a completed full checkpoint")
        return model, completed, best, history
    stop = min(CONFIG["epochs"], completed + int(CONFIG["epochs_per_run"]))
    for epoch in range(completed, stop):
        offset = int(partial["step"]) if partial and int(partial["epoch"]) == epoch else 0
        TRAIN_LOADER.sampler.set_epoch(epoch, offset * CONFIG["batch_size"])
        TRAIN_LOADER.dataset.epoch = epoch
        TRAIN_LOADER.generator.manual_seed(CONFIG["seed"] + epoch)
        set_seed(CONFIG["seed"] + epoch)
        if offset:
            torch.set_rng_state(resumed["torch_rng_state"])
            if DEVICE.type == "cuda" and resumed.get("cuda_rng_state") is not None:
                torch.cuda.set_rng_state(resumed["cuda_rng_state"], DEVICE)
        sums = dict(partial.get("sums", {})) if offset else {}
        seen = int(partial.get("seen", 0)) if offset else 0
        prior_seconds = float(partial.get("seconds", 0)) if offset else 0
        partial = None
        started = last_save = time.time()
        model.train()
        if CONFIG["freeze_backbone_bn"]:
            freeze_backbone_bn(model)
        optimizer.zero_grad(set_to_none=True)
        for i, batch in enumerate(tqdm(TRAIN_LOADER, desc=f'epoch {epoch+1}', leave=True)):
            step = offset + i
            images = batch["image"].to(DEVICE, non_blocking=True)
            labels = batch["label"].to(DEVICE, non_blocking=True)
            masks = batch["mask"].to(DEVICE, non_blocking=True)
            valid = batch["valid_mask"].to(DEVICE, non_blocking=True)
            trusted = batch["seg_valid"].to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, enabled=amp):
                outputs = model(images, valid_mask=valid)
                losses = criterion(outputs, labels, masks, valid_mask=valid, seg_valid=trusted)
                group_size = min(accum, full_batches - (step // accum) * accum)
                loss = losses["total"] / group_size
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Nonfinite loss at epoch {epoch+1}, batch {step+1}")
            scaler.scale(loss).backward()
            boundary = (step + 1) % accum == 0 or step + 1 == full_batches
            if boundary:
                scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale() >= scale:
                    scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            size = images.shape[0]
            seen += size
            for key in ("total", "classification", "segmentation_bce", "segmentation_dice"):
                sums[key] = sums.get(key, 0.0) + float(losses[key].detach()) * size
            for key in ("segmentation_valid_samples", "segmentation_positive_samples", "segmentation_ignored_fake_samples"):
                sums[key] = sums.get(key, 0.0) + float(losses[key])
            elapsed = time.time() - NOTEBOOK_START_TIME
            if i + 1 == min(10, max(full_batches - offset, 1)):
                rate = (i + 1) * CONFIG["batch_size"] / max(time.time() - started, 1e-6)
                throughput = dict(epoch=epoch+1, next_step=step+1, images_per_second=rate,
                                  estimated_training_only_epoch_hours=len(TRAIN_LOADER.dataset)/rate/3600,
                                  excludes_validation_and_setup=True)
                (WORKING / "logs/throughput.json").write_text(json.dumps(throughput, indent=2), encoding="utf-8")
                print("TRAIN_THROUGHPUT", json.dumps(throughput))
            out_of_time = elapsed >= CONFIG["time_budget_hours"] * 3600
            if boundary and (out_of_time or time.time() - last_save >= CONFIG["mid_epoch_checkpoint_minutes"] * 60):
                state = dict(epoch=epoch, step=step+1, sums=sums, seen=seen,
                             seconds=prior_seconds+time.time()-started)
                save_checkpoint(last_path, model, optimizer, scaler, scheduler, epoch, best, history, state)
                last_save = time.time()
                if out_of_time:
                    print("PARTIAL_EPOCH_SAVED", json.dumps(state))
                    return model, epoch, best, history
            del images, labels, masks, valid, trusted, outputs, losses, loss
        validation, _ = evaluate_loader(model, VAL_LOADER, "val")
        row = dict(epoch=epoch+1, seconds=prior_seconds+time.time()-started,
                   **{"train_"+k: v/max(seen, 1) for k, v in sums.items()
                      if not k.startswith("segmentation_") or k.endswith(("bce", "dice"))},
                   **{"val_"+k: v for k, v in validation.items()},
                   segmentation_valid_samples=sums.get("segmentation_valid_samples", 0),
                   segmentation_ignored_fake_samples=sums.get("segmentation_ignored_fake_samples", 0),
                   backbone_lr=optimizer.param_groups[0]["lr"],
                   heads_lr=optimizer.param_groups[-1]["lr"])
        history.append(row)
        score = float(validation["accuracy"])
        if score >= best:
            best = score
            save_checkpoint(best_path, model, optimizer, scaler, scheduler, epoch+1, best, history)
        save_checkpoint(last_path, model, optimizer, scaler, scheduler, epoch+1, best, history)
        pd.DataFrame(history).to_csv(WORKING / "logs/validation_history.csv", index=False)
        print("EPOCH_COMPLETE", json.dumps(row))
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    return model, stop, best, history


@torch.no_grad()
def benchmark_inference(model):
    batch = next(iter(VAL_LOADER))
    images, valid = batch["image"].to(DEVICE), batch["valid_mask"].to(DEVICE)
    model.eval()
    model(images, valid_mask=valid)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    for _ in range(3):
        model(images, valid_mask=valid)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    milliseconds = (time.perf_counter()-started) * 1000 / 3
    result = dict(batch_size=len(images), image_size=CONFIG["image_size"],
                  batch_latency_ms=milliseconds, per_image_ms_at_batch_size=milliseconds/len(images),
                  peak_allocated_vram_bytes=torch.cuda.max_memory_allocated() if DEVICE.type == "cuda" else None,
                  device=torch.cuda.get_device_name() if DEVICE.type == "cuda" else "cpu",
                  torch_version=torch.__version__, inference_precision="float32", includes_preprocessing=False,
                  repeats=3, warmup_batches=1)
    (WORKING / "logs/inference_benchmark.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
    print("INFERENCE_BENCHMARK", json.dumps(result))
    return result


def run_notebook():
    model, done, best, history = train_one_experiment()
    state = dict(experiment=CONFIG["experiment"], epochs_completed=done, epochs_total=CONFIG["epochs"],
                 protocol_hash=PROTOCOL_HASH, model_source_hash=MODEL_SOURCE_HASH, exp_cfg=EXP_CFG)
    (WORKING / "logs/run_state.json").write_text(json.dumps(state, indent=2), encoding="utf-8")
    if done < CONFIG["epochs"]:
        print("RUN_SAVED_FOR_CONTINUATION", json.dumps(state))
        return state
    best_path = WORKING / "checkpoints" / f'{CONFIG["experiment"]}_best.pt'
    selected = load_checkpoint(best_path, model)
    state["best_epoch"] = int(selected["epoch"])
    del selected
    if CONFIG["run_mode"] == "smoke":
        metrics, _ = evaluate_loader(model, VAL_LOADER, "val")
        assert metrics["n"] == 128 and len(MASKS) == 190335
        assert all(math.isfinite(metrics[k]) for k in ("accuracy", "f1", "precision", "recall"))
        assert done == 1
        state.update(split="val", metrics=metrics, status="SMOKE_PASSED")
        (WORKING / "smoke_results.json").write_text(json.dumps(state, indent=2), encoding="utf-8")
        if CONFIG.get("benchmark_inference", False):
            benchmark_inference(model)
        print("TRAIN_SMOKE_VALIDATION_PASSED", CONFIG["experiment"])
        return state
    if time.time() - NOTEBOOK_START_TIME >= CONFIG["time_budget_hours"] * 3600:
        print("TRAINING_COMPLETE_TEST_PENDING_TIME_BUDGET", json.dumps(state))
        return state
    metrics, records = evaluate_loader(model, TEST_LOADER, "test")
    if CONFIG.get("benchmark_inference", False):
        metrics.update(benchmark_inference(model))
    pd.DataFrame(records).to_csv(WORKING / "predictions" / f'{CONFIG["experiment"]}_predictions.csv', index=False)
    row = dict(**state, **metrics, split="test", run_mode="full", implementation_version="ow_v3", status="ok")
    row["exp_cfg"] = json.dumps(EXP_CFG, sort_keys=True)
    pd.DataFrame([row]).to_csv(WORKING / "experiment_results.csv", index=False)
    print("FINAL_TEST", json.dumps(row))
    return row

RUN_RESULT = run_notebook()


## Research interpretation

Accuracy/F1 use Wild folder labels. Fake Dice/IoU measure teacher agreement on
available targets; Real positive-mask area is reported separately. All variants
share the training protocol; architecture flags distinguish checkpoints. Keep
Test out of tuning. A higher score must be demonstrated, not assumed.
